# Monthly Job-Advertisement Index Forecasting

Predict the next month's overall index using previously available
observations. Evaluate chronologically using a fixed historical
data snapshot, acknowledging publication delays and revisions.

In [12]:
from pathlib import Path

import pandas as pd

monthly = pd.read_csv(
    Path("../data/processed/monthly_jobs_online.csv"),
    parse_dates=["Date"],
    index_col="Date"
)

monthly.head(10)

,overall_index,reported_yoy_pct,SkilledIndex,UnskilledIndex,Auckland,Wellington,North Island Other,Canterbury,South Island Other,Business services,...,Clerical and Administrative Workers,Sales Workers,Machinery Operators and Drivers,Labourers,Highly-Skilled,Skilled,Semi-Skilled,Low-Skilled,Unskilled,calculated_yoy_pct
Date,,,,,,,,,,,,,,,,,,,,,
2007-05-01,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN
2007-06-01,92.1,0.0,91.5,92.5,91.8,93.5,94.4,94.1,76.6,91.6,...,95.2,93.8,80.2,89.2,91.2,92.8,89.6,92.8,91.2,NaN
2007-07-01,95.7,0.0,95.3,94.9,96.4,95.1,95.5,98.6,84.0,93.2,...,95.9,98.7,92.3,89.5,95.1,93.7,101.7,94.1,97.4,NaN
2007-08-01,103.2,0.0,102.0,103.1,101.2,98.8,113.3,109.5,106.4,96.9,...,103.5,98.6,109.6,110.8,97.7,110.8,111.6,103.1,102.8,NaN
2007-09-01,97.0,0.0,96.7,96.4,93.9,95.1,105.7,103.9,107.4,84.8,...,90.8,96.9,104.7,103.5,91.1,112.1,98.8,95.3,100.2,NaN
2007-10-01,102.2,0.0,104.6,97.0,96.3,101.0,118.6,110.5,119.9,85.9,...,91.4,93.3,108.0,107.0,97.5,126.5,100.8,96.4,99.0,NaN
2007-11-01,101.1,0.0,106.6,90.2,96.2,98.8,118.1,102.9,122.9,88.0,...,87.1,82.5,108.4,99.2,103.2,120.7,94.9,90.4,89.3,NaN
2007-12-01,70.0,0.0,75.0,60.3,65.1,67.8,83.7,71.1,105.8,59.4,...,57.0,55.3,65.4,71.2,75.2,78.7,62.8,60.6,58.9,NaN
2008-01-01,101.8,0.0,101.9,98.6,97.1,97.0,113.3,119.2,117.4,93.1,...,94.3,89.4,109.0,119.5,104.8,91.5,107.2,96.6,105.3,NaN


In [11]:
target = monthly["overall_index"].asfreq("MS")

assert target.notna().all(), "Missing monthly target values"

target.head()

Date
2007-05-01    100.0
2007-06-01     92.1
2007-07-01     95.7
2007-08-01    103.2
2007-09-01     97.0
Freq: MS, Name: overall_index, dtype: float64

### Reserve validation and test periods

In [2]:
train_target = target.loc[:"2022-07-01"]
validation_target = target.loc["2022-08-01":"2024-07-01"]
test_target = target.loc["2024-08-01":"2026-07-01"]

for name, series in {
    "Training": train_target,
    "Validation": validation_target,
    "Test": test_target
}.items():
    print(
        name,
        len(series),
        series.index.min().to_period("M"),
        series.index.max().to_period("M")
    )

Training 183 2007-05 2022-07
Validation 24 2022-08 2024-07
Test 24 2024-08 2026-07


### Create two baseline forecasts

In [13]:
baseline_forecasts = pd.DataFrame({
    "actual": target,
    "last_month_naive": target.shift(1),
    "seasonal_naive": target.shift(12)
})

validation_baselines = baseline_forecasts.loc[
    validation_target.index
].copy()

assert validation_baselines.notna().all().all()

validation_baselines.head()

,actual,last_month_naive,seasonal_naive
Date,,,
2022-08-01,215.5,196.4,168.4
2022-09-01,202.9,215.5,176.8
2022-10-01,190.0,202.9,176.5
2022-11-01,178.7,190.0,184.7
2022-12-01,116.7,178.7,141.1


In [9]:
validation_errors = pd.DataFrame({
    "last_month_naive": (
        validation_baselines["last_month_naive"]
        - validation_baselines["actual"]
    ).abs(),

    "seasonal_naive": (
        validation_baselines["seasonal_naive"]
        - validation_baselines["actual"]
    ).abs()
})

validation_errors.head()

,last_month_naive,seasonal_naive
Date,,
2022-08-01,19.1,47.1
2022-09-01,12.6,26.1
2022-10-01,12.9,13.5
2022-11-01,11.3,6.0
2022-12-01,62.0,24.4


In [10]:
validation_mae = validation_errors.mean().sort_values()

validation_mae.round(2)

last_month_naive    21.50
seasonal_naive      40.21
dtype: float64

In [14]:
validation_rmse = (validation_errors.pow(2).mean()) ** 0.5

In [15]:
baseline_metrics = pd.DataFrame({
    "MAE": validation_mae,
    "RMSE": validation_rmse
}).sort_values("MAE")

baseline_metrics.round(2)

,MAE,RMSE
last_month_naive,21.50,27.12
seasonal_naive,40.21,43.03


### Forecasting design and baseline results

The task is to predict the next month's overall job-advertisement
index using observations available through the preceding month,
once published.

Data is split chronologically:
- Training: May 2007–July 2022
- Validation: August 2022–July 2024
- Test: August 2024–July 2026

Validation uses rolling one-month-ahead predictions. Each prediction
may use newly observed prior-month values, but not the target month's
actual value.

| Baseline | Validation MAE | Validation RMSE |
|---|---:|---:|
| Last-month naïve | 21.50 | 27.12 |
| Seasonal naïve | 40.21 | 43.03 |

Errors are measured in index points. Last-month naïve performed
better on both metrics during validation. This does not establish
that seasonality is absent or guarantee future performance.

The test period has not been evaluated. Results use one historical
data snapshot and do not fully reproduce historical release timing
or data revisions.